# Topic 01: Document Loading of the RAG pipeline.

In production systems, "Vanilla RAG" fails primarily because it treats documents as flat text files. Enterprise documents are messy: they contain multi-column layouts, tables, headers, footers, charts, and embedded footnotes that break unstructured text extraction.

## 01. Document Loading & Layout Parsing
🎯 The Problem
When you read a PDF using basic text extractors (like standard pypdf), text streams out sequentially based on underlying coordinate streams, often mixing up columns, injecting page numbers into the middle of sentences, or completely shredding markdown tables into unreadable strings.


🛠️ Production Tech Stack
**LlamaParse:** Cloud-native, LLM-augmented parser designed explicitly to convert complex PDFs and tables into clean Markdown.

**Unstructured.io:** Open-source ingestion engine that classifies elements (Title, NarrativeText, Table, Image) across PDFs, HTML, Word docs, and emails.

**PyMuPDF (fitz):** High-performance C-backed Python library for fast text, image, and metadata extraction.

### Code Implementation: Layout-Aware Ingestion Script
Create this file in your repository under: 07-RAG/01_document_loading/load_documents.py

In [ ]:
"""
Module 01: Document Loading & Parsing
Demonstrates layout-aware loading using PyMuPDF and structuring text for RAG ingestion.
"""

import os
from typing import List, Dict, Any
# import fitz  # PyMuPDF library

class DocumentLoader:
    def __init__(self, file_path: str):
        self.file_path = file_path

    def load_pdf_with_metadata(self) -> List[Dict[str, Any]]:
        """
        Loads a PDF document page by page, extracting structural metadata 
        such as page numbers and block types to preserve context.
        """
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")

        extracted_pages = []
        
        # --- Simulated execution pattern using PyMuPDF (fitz) ---
        # doc = fitz.open(self.file_path)
        # for page_num, page in enumerate(doc):
        #     text = page.get_text("text")
        #     
        #     # Clean common parsing anomalies (e.g., repeating headers/footers)
        #     cleaned_text = self._clean_boilerplate(text)
        #     
        #     extracted_pages.append({
        #         "page_number": page_num + 1,
        #         "content": cleaned_text,
        #         "source": self.file_path,
        #         "total_chars": len(cleaned_text)
        #     })
        
        # Mock payload for demonstration structure
        extracted_pages.append({
            "page_number": 1,
            "content": "Mock parsed text block from enterprise financial statement...",
            "source": self.file_path,
            "total_chars": 58
        })

        return extracted_pages

    def _clean_boilerplate(self, text: str) ->- str:
        """Strips running headers, footers, and artifact noise from pages."""
        lines = text.split("\n")
        # Filter out lines that look like page numbers or common headers
        filtered_lines = [line for line in lines if not line.strip().isdigit()]
        return "\n".join(filtered_lines)


# Example execution block
if __name__ == "__main__":
    loader = DocumentLoader(file_path="sample_annual_report.pdf")
    try:
        pages = loader.load_pdf_with_metadata()
        print(f"Successfully loaded {len(pages)} page(s) with structural metadata.")
        print("Sample Page Object:", pages[0])
    except FileNotFoundError as e:
        print(e)

### Key Architectural Takeaways
**Preserve Metadata Early:** Always bind the page_number, source_file, and section_title directly to the document object during load time. Downstream vector searches rely heavily on this for filtering.

**Table Preservation:** If your document contains tabular financials or specs, force your loader to extract them into Markdown tables rather than raw strings. LLMs reason over Markdown pipes (| Column |) significantly better than raw spacing.